# Pipeline Metadata-Driven — Fase 3 (CRESA)

Notebook genérico para Silver/Gold sobre el catálogo `dlh_cresa` ya productivo.
No usar para Bronze: las 173 tablas Bronze ya tienen su propio mecanismo de ingesta
(OData Dynamics 365, Fivetran, batch) — este notebook solo conforma (Silver) y
certifica (Gold) sobre lo que ya existe.

Parámetros esperados (widgets): `config_path`, `environment`, `layer` (`silver`|`gold`), `run_mode`, `dry_run`.

In [ ]:
dbutils.widgets.text("config_path", "", "Ruta YAML")
dbutils.widgets.text("environment", "prod", "Ambiente (unico real hoy en CRESA)")
dbutils.widgets.dropdown("layer", "silver", ["silver", "gold"], "Capa")
dbutils.widgets.dropdown("run_mode", "delta", ["initial", "delta", "reprocess"], "Modo de ejecucion")
dbutils.widgets.dropdown("dry_run", "false", ["true", "false"], "Dry run")

config_path = dbutils.widgets.get("config_path")
environment = dbutils.widgets.get("environment")
layer = dbutils.widgets.get("layer")
run_mode = dbutils.widgets.get("run_mode")
dry_run = dbutils.widgets.get("dry_run") == "true"

assert config_path, "config_path es obligatorio"

In [ ]:
import yaml
import hashlib
from datetime import datetime

CATALOG = "dlh_cresa"  # catalogo unico ya productivo - no se crean catalogos nuevos
GOVERNANCE_SCHEMA = "audit01"  # se extiende el esquema de auditoria ya existente

with open(config_path) as f:
    cfg = yaml.safe_load(f)

batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print(f"Ejecutando layer={layer} run_mode={run_mode} dry_run={dry_run} batch_id={batch_id}")
print(f"Config: {cfg.get('entity_name') or cfg.get('product_name')}")

## 1. Leer fuentes

Las fuentes de Silver son tablas Bronze/Gold **ya existentes** en `dlh_cresa`
(no se crean tablas Bronze nuevas desde este notebook).

In [ ]:
source_frames = {}
for src in cfg["sources"]:
    table_fqn = src["table"]
    alias = src["alias"]
    required = src.get("required", True)
    try:
        df = spark.table(table_fqn)
        source_frames[alias] = df
        print(f"OK  {alias}: {table_fqn} ({df.count()} filas)")
    except Exception as e:
        if required:
            raise
        print(f"WARN fuente opcional no disponible: {table_fqn} — {e}")

## 2. Conformar / aplicar reglas de la capa

En Silver: estandarizar, resolver llave de negocio, aplicar reglas de supervivencia,
marcar calidad (`_quality_status`, `_quality_flags`) sin descartar registros.

En Gold: aplicar reglas de certificación, calcular campos derivados
(ej. `cupo_disponible`, `flag_contactable`) y asignar `_certification_status`.

In [ ]:
def apply_rules(df, rules, default_action="flag_only"):
    """Aplica reglas declaradas en YAML. No descarta filas salvo accion explicita reject_from_product."""
    from pyspark.sql import functions as F

    flags_col = F.array().alias("_quality_flags")
    df = df.withColumn("_quality_flags", F.array())

    for rule in rules:
        rule_id = rule["rule_id"]
        rtype = rule["type"]
        action = rule.get("action", default_action)

        if rtype == "not_null":
            cond = F.lit(False)
            for col in rule["columns"]:
                cond = cond | F.col(col).isNull()
            df = df.withColumn(
                "_quality_flags",
                F.when(cond, F.array_union(F.col("_quality_flags"), F.array(F.lit(rule_id))))
                 .otherwise(F.col("_quality_flags"))
            )
            if action == "reject_from_product":
                df = df.filter(~cond)

        elif rtype in ("pending_business_rule", "cross_source_reconciliation"):
            # Regla de negocio aun no definida (ver analisis_caracterizacion/06_hipotesis_validacion.md).
            # Se deja el campo marcado, sin bloquear el pipeline.
            df = df.withColumn(
                "_quality_flags",
                F.array_union(F.col("_quality_flags"), F.array(F.lit(rule_id)))
            )

    df = df.withColumn(
        "_quality_status",
        F.when(F.size(F.col("_quality_flags")) > 0, F.lit("requires_review")).otherwise(F.lit("ok"))
    )
    return df

In [ ]:
rules_key = "cleansing_rules" if layer == "silver" else "certification_rules"
rules = cfg.get(rules_key, {}).get("rules", [])
default_action = cfg.get(rules_key, {}).get("action_default", "flag_only")

# El desarrollador reemplaza esto por la logica real de join/derivacion segun el YAML de la entidad/producto.
base_alias = cfg["sources"][0]["alias"]
result_df = source_frames[base_alias]
result_df = apply_rules(result_df, rules, default_action)

## 3. Metadata de gobierno y persistencia

In [ ]:
from pyspark.sql import functions as F

target = cfg["target"] if layer == "silver" else cfg["outputs"][0]
target_fqn = f"{target.get('catalog', CATALOG)}.{target['schema']}.{target['table']}"

if layer == "silver":
    result_df = (
        result_df
        .withColumn("_domain_official", F.lit(cfg["domain_official"]))
        .withColumn("_business_key", F.lit(",".join(cfg["business_key"]["columns"])))
        .withColumn("_silver_batch_id", F.lit(batch_id))
    )
else:
    result_df = (
        result_df
        .withColumn("_certification_status", F.lit(cfg.get("status", "temporal")))
        .withColumn("_data_product_name", F.lit(cfg["product_name"]))
        .withColumn("_publication_timestamp", F.current_timestamp())
    )

print(f"Destino: {target_fqn} | filas: {result_df.count()} | dry_run={dry_run}")

if not dry_run:
    (result_df.write
        .format("delta")
        .mode("overwrite" if run_mode == "initial" else "append")
        .saveAsTable(target_fqn))
    print(f"Escrito en {target_fqn}")
else:
    print("Dry run — no se escribio nada.")

## 4. Registrar ejecución en control plane (`audit01`)

In [ ]:
control_table = (
    f"{CATALOG}.{GOVERNANCE_SCHEMA}.gobierno_silver_resultados"
    if layer == "silver"
    else f"{CATALOG}.{GOVERNANCE_SCHEMA}.gobierno_gold_certificacion"
)

run_record = spark.createDataFrame([{
    "batch_id": batch_id,
    "config_path": config_path,
    "layer": layer,
    "target_table": target_fqn,
    "run_mode": run_mode,
    "dry_run": dry_run,
    "executed_at": datetime.utcnow().isoformat(),
}])

if not dry_run:
    try:
        run_record.write.format("delta").mode("append").saveAsTable(control_table)
        print(f"Ejecucion registrada en {control_table}")
    except Exception as e:
        print(f"AVISO: tabla de control {control_table} no existe todavia — crearla segun 10_metodologia_lakehouse_gobierno_fase3.md. Detalle: {e}")